In [ ]:
import os
import numpy as np
import cv2
import pydicom
import matplotlib.pyplot as plt
import tensorflow as tf
import random
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import shap
import kagglehub

# --- IMAGE LOADING & PREPROCESSING ---

def load_and_preprocess_single_image(img_path):
    """Load and preprocess a single DICOM or JPEG image"""
    try:
        if img_path.lower().endswith('.dcm'):
            dcm = pydicom.dcmread(img_path)
            image = dcm.pixel_array.astype(float)
            window_center, window_width = 40, 80
            image = np.clip(image, window_center - window_width // 2, window_center + window_width // 2)
            image = np.stack([image] * 3, axis=-1)
        else:
            image = cv2.imread(img_path)
            if image is None:
                raise ValueError(f"Failed to load image: {img_path}")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Normalize to [0,1]
        image = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-7)
        image = image.astype(np.float32)

        # Resize for model input
        processed_img = cv2.resize(image, (224, 224))
        return processed_img, image
    except Exception as e:
        print(f"Error loading {img_path}: {e}")
        return None, None

def load_dataset(base_path, max_samples_per_class=100, random_seed=42):
    np.random.seed(random_seed)
    processed_images, original_images, paths, labels = [], [], [], []

    # Get aneurysm images
    aneurysm_path = os.path.join(base_path, 'aneurysm')
    aneurysm_files = [(os.path.join(aneurysm_path, f), 1)
                      for f in os.listdir(aneurysm_path)
                      if f.endswith(('.jpg', '.dcm'))] if os.path.exists(aneurysm_path) else []

    # Get control (non-aneurysm) images
    non_aneurysm_files = []
    for category in ['tumor', 'cancer']:
        cat_path = os.path.join(base_path, category)
        if os.path.exists(cat_path):
            non_aneurysm_files.extend([(os.path.join(cat_path, f), 0)
                                       for f in os.listdir(cat_path)
                                       if f.endswith(('.jpg', '.dcm'))])

    # Sample images
    selected = (
    random.sample(aneurysm_files, min(len(aneurysm_files), max_samples_per_class)) +
    random.sample(non_aneurysm_files, min(len(non_aneurysm_files), max_samples_per_class))
    )

    for img_path, label in selected:
        proc, orig = load_and_preprocess_single_image(img_path)
        if proc is not None:
            processed_images.append(proc)
            original_images.append(orig)
            paths.append(img_path)
            labels.append(label)

    return (np.array(processed_images), np.array(original_images),
            np.array(paths), np.array(labels))


# --- MODEL DEFINITION & TRAINING ---

def create_model():
    """Create EfficientNetB0-based binary classifier"""
    base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    inputs = layers.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def train_model(model, train_images, train_labels, val_images, val_labels):
    history = model.fit(train_images, train_labels,
                        validation_data=(val_images, val_labels),
                        epochs=10, batch_size=32)
    return history

def evaluate_model(model, test_images, test_labels):
    loss, acc = model.evaluate(test_images, test_labels)
    print(f"Test Loss: {loss:.4f}, Accuracy: {acc:.4f}")

# --- SHAP EXPLANATIONS ---

def generate_shap_explanations(model, test_images, background_images, num_samples=3):
    """Generate SHAP explanations for a few test samples"""
    test_subset = test_images[:num_samples]
    background_subset = background_images[:20]  # Keep this small!

    print(f"Running SHAP on {num_samples} samples...")

    explainer = shap.DeepExplainer(model, background_subset)
    shap_values = explainer.shap_values(test_subset)

    # Show SHAP results per image
    shap.image_plot(shap_values, test_subset)


# --- MAIN ---

def main():
    dataset_path = kagglehub.dataset_download("trainingdatapro/computed-tomography-ct-of-the-brain")
    base_path = os.path.join(dataset_path, 'files')

    # Load and split data
    processed, original, paths, labels = load_dataset(base_path)
    X_train, X_temp, y_train, y_temp = train_test_split(processed, labels, test_size=0.3, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

    # Build and train model
    model = create_model()
    train_model(model, X_train, y_train, X_val, y_val)
    evaluate_model(model, X_test, y_test)

    # SHAP explanations
    generate_shap_explanations(model, X_test, X_train)

if __name__ == "__main__":
    main()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 104s 9s/step - accuracy: 0.7660 - loss: 0.4357 - val_accuracy: 0.4667 - val_loss: 0.7422
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 87s 9s/step - accuracy: 0.9892 - loss: 0.0224 - val_accuracy: 0.4667 - val_loss: 0.7022
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 49s 10s/step - accuracy: 0.9946 - loss: 0.0215 - val_accuracy: 0.3000 - val_loss: 0.6960
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 78s 9s/step - accuracy: 1.0000 - loss: 0.0093 - val_accuracy: 0.4667 - val_loss: 0.6932
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 84s 9s/step - accuracy: 1.0000 - loss: 0.0025 - val_accuracy: 0.5333 - val_loss: 0.6993
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 80s 9s/step - accuracy: 1.0000 - loss: 1.5791e-04 - val_accuracy: 0.5333 - val_loss: 0.7881
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 82s 9s/step - accuracy: 1.0000 - loss: 1.0886e-04 - val_accuracy: 0.5333 - val_loss: 0.9224
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 81s 9s/step - accuracy: 1.0000 - lo

/usr/local/lib/python3.11/dist-packages/shap/explainers/_deep/deep_tf.py:94: UserWarning: Your TensorFlow version is newer than 2.4.0 and so graph support has been removed in eager mode and some static graphs may not be supported. See PR #1483 for discussion.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor_238
Received: inputs=['Tensor(shape=(20, 224, 224, 3))']
  warnings.warn(msg)
/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor_238
Received: inputs=['Tensor(shape=(40, 224, 224, 3))']
  warnings.warn(msg)


StagingError: in user code:

    File "/usr/local/lib/python3.11/dist-packages/shap/explainers/_deep/deep_tf.py", line 265, in grad_graph  *
        x_grad = tape.gradient(out, shap_rAnD)

    LookupError: gradient registry has no entry for: shap_DepthwiseConv2dNative


In [ ]:
!pip install pydicom


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 16.2 MB/s eta 0:00:00
